# Dev notebook — phase3_aggregate.py

Chạy từng bước Pha 3 (tổng hợp + xếp hạng) trên output thật của Pha 2. Notebook này **import thẳng từ `phase3_aggregate.py`** — không định nghĩa lại logic.

Cần có `/tmp/aqi` từ trước (chạy `phase1_clean.py` rồi `phase2_aqi.py`, hoặc dùng 2 notebook dev tương ứng).

Sửa `phase3_aggregate.py` xong, chạy lại cell là thấy ngay nhờ `%autoreload 2`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from pyspark.sql import SparkSession

from phase3_aggregate import compute_daily, compute_ranking, print_top_ranking

spark = (
    SparkSession.builder.appName("phase3_dev").master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")  # bắt buộc, xem ghi chú trong phase1_clean.py
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

AQI_INPUT = "/tmp/aqi"  # output của phase2_aqi.py

## Bước 0 — đọc output Pha 2

In [ ]:
df = spark.read.parquet(AQI_INPUT)
print("so dong:", df.count())
df.select("city", "country", "dt", "aqi", "aqi_level", "dominant_pollutant").show(5)

## Bước 1 — `compute_daily()`: trung bình/max AQI + phân bố 6 mức theo (city, dt)

In [ ]:
daily = compute_daily(df)
print("so dong (city x ngay):", daily.count())
daily.filter(F.col("city") == "Ho Chi Minh City").orderBy("dt").show(10)

Kiểm tra: tổng `count_level_1..6` phải bằng đúng `n_hours` cho mọi dòng (không đếm thiếu/thừa giờ nào).

In [ ]:
level_cols = [f"count_level_{i}" for i in range(1, 7)]
check = daily.withColumn("tong_level", sum(F.col(c) for c in level_cols))
mismatch = check.filter(F.col("tong_level") != F.col("n_hours"))
print("so dong lech (ky vong 0):", mismatch.count())

## Bước 2 — `compute_ranking()`: xếp hạng thành phố theo avg_aqi (rank 1 = ô nhiễm nhất)

In [ ]:
ranking = compute_ranking(daily)
print_top_ranking(ranking, n=10)

Kiểm tra: mỗi ngày không được có 2 thành phố cùng 1 rank.

In [ ]:
dup = (
    ranking.groupBy("dt", "rank").count().filter(F.col("count") > 1)
)
print("so cap (dt, rank) bi trung (ky vong 0):", dup.count())

## Bước 3 — xem xu hướng rank của 1 thành phố qua thời gian

In [ ]:
ranking.filter(F.col("city") == "New Delhi").orderBy("dt").select("dt", "rank", "avg_aqi", "worst_pollutant").show(15)

## Ghi chú

- Notebook này chỉ để đọc/hiểu/debug — sửa logic thì sửa `phase3_aggregate.py`.
- Ghi thử ra parquet thật: `python jobs/phase3_aggregate.py --input /tmp/aqi --output /tmp/agg`.